# VisiPruner trên SmolVLM · Deep Discard toàn bộ test

Chạy baseline và bảy lượt **Discard từ từng layer 24–30 trở đi** trên **16.817 câu DriveLM test**. Layer đánh số từ 1. Cả 8 cấu hình dùng cùng checkpoint và greedy decoding với trần 512 token.

Bật Internet và GPU. Sau baseline, notebook chạy hai lượt Discard cùng lúc trên T4 ×2, hoặc chạy tuần tự trên một GPU. Lượt layer 30 chạy riêng. Log tiến độ lưu theo cấu hình trong `logs/`.

In [ ]:
%%writefile /kaggle/working/visipruner-deep-full.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export HF_TOKEN_SECRET=''
OUTPUT_DIR=/kaggle/working/visipruner_deep_discard_512_outputs
RESUME_FROM=''
BATCH_SIZE=8
MAX_NEW_TOKENS=512


## Cài đặt

Cập nhật mã nguồn và cài thư viện chạy mô hình, chấm điểm và vẽ hình.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt" pycocoevalcap==1.2
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq default-jre-headless
fi


## Khôi phục phiên trước (tùy chọn)

Nếu chia qua nhiều phiên, lưu thư mục đầu ra, gắn bản lưu làm Kaggle Input và đặt `RESUME_FROM` thành thư mục chứa `manifest.json`. Cell dưới sao chép vào thư mục đầu ra mới. Với đầu ra đã có trong phiên hiện tại, chạy lại các cell đo để tiếp tục từ bộ ảnh cuối đã lưu.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
if [ -n "$RESUME_FROM" ] && [ ! -e "$OUTPUT_DIR" ]; then
  test -f "$RESUME_FROM/manifest.json"
  mkdir -p "$(dirname "$OUTPUT_DIR")"
  staging=$(mktemp -d "${OUTPUT_DIR}.restore.XXXXXX")
  trap 'rm -rf -- "$staging"' EXIT
  cp -R -- "$RESUME_FROM"/. "$staging/"
  chmod -R u+rwX -- "$staging"
  mv -T -- "$staging" "$OUTPUT_DIR"
  trap - EXIT
fi


## Chuẩn bị dữ liệu

Dùng toàn bộ split test, giữ nguyên thứ tự và ID câu hỏi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.mvpruner_ablation fetch-hf --output "$DATA_DIR"
python -u -m smolvlm_attention.visipruner_ablation prepare --suite deep-full --test "$DATA_DIR/multi_frame/multi_frame_test.json" --image-root "$DATA_DIR" --output "$OUTPUT_DIR"


## 01/08 · Baseline

Giữ đủ token ảnh, kiểm tra khớp với decoder gốc và lưu cache đặc trưng dùng chung cho các cặp sau.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant baseline --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 02–03/08 · Discard layer 24 và 25

- **Discard 24** loại token ảnh khỏi chuỗi và KV cache từ layer 24 đến hết.
- **Discard 25** loại token ảnh khỏi chuỗi và KV cache từ layer 25 đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-24 deep-discard-25 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 04–05/08 · Discard layer 26 và 27

- **Discard 26** loại token ảnh khỏi chuỗi và KV cache từ layer 26 đến hết.
- **Discard 27** loại token ảnh khỏi chuỗi và KV cache từ layer 27 đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-26 deep-discard-27 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 06–07/08 · Discard layer 28 và 29

- **Discard 28** loại token ảnh khỏi chuỗi và KV cache từ layer 28 đến hết.
- **Discard 29** loại token ảnh khỏi chuỗi và KV cache từ layer 29 đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-28 deep-discard-29 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 08/08 · Discard layer 30

- **Discard 30** loại token ảnh khỏi chuỗi và KV cache từ layer 30 đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-discard-30 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Chấm điểm và vẽ hình

Xuất bảng điểm và hai hình PNG/PDF cho ΔOverall và ΔYes/No accuracy so với baseline toàn tập.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation score --output "$OUTPUT_DIR"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-deep-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation plot --output "$OUTPUT_DIR"
